# 1. Setup & Data Loading

This notebook performs Exploratory Data Analysis (EDA) on the **Student Burnout & Dropout Risk** dataset.  
We explore demographic, academic, lifestyle, and mental health features and how they relate to  
**Dropout Risk** and **Burnout Level**.


## 1.1 Import Libraries


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Global style settings
sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.1)
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 100,
})

print("✅ All libraries loaded successfully.")

## 1.2 Load Dataset


In [ ]:
df = pd.read_csv('data.csv')
print(f"Dataset shape: {df.shape}")
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")

## 1.3 First Look at the Data


In [ ]:
df.head(10)

## 1.4 Data Types & Info


In [ ]:
df.info()

## 1.5 Statistical Summary (Numeric)


In [ ]:
df.describe().round(2)

## 1.6 Statistical Summary (Categorical)


In [ ]:
df.describe(include='object')

## 1.7 Unique Value Counts for Categorical Columns


In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    print(f"\n📌 {col} ({df[col].nunique()} unique):")
    print(df[col].value_counts().to_string())

---

# 2. Missing Data Analysis

Understanding missing data patterns is crucial before any modeling or deeper analysis.


## 2.1 Missing Value Counts


In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct,
    'Dtype': df.dtypes
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)

print(f"Total missing values: {missing.sum():,} across {len(missing_df)} columns\n")
print(missing_df.to_string())

## 2.2 Missing Data Heatmap


In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='YlOrRd', ax=ax)
ax.set_title('Missing Data Heatmap (Yellow = Present, Red = Missing)', fontsize=14, pad=12)
ax.set_xlabel('Features')
plt.tight_layout()
plt.show()

## 2.3 Missing Data Matrix by Column


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
missing_sorted = df.isnull().sum().sort_values(ascending=True)
missing_sorted = missing_sorted[missing_sorted > 0]
colors = ['#e74c3c' if v > 50 else '#f39c12' if v > 10 else '#2ecc71' for v in missing_sorted.values]
bars = ax.barh(missing_sorted.index, missing_sorted.values, color=colors, edgecolor='white', height=0.6)
for bar, val in zip(bars, missing_sorted.values):
    ax.text(val + 2, bar.get_y() + bar.get_height()/2, str(val), va='center', fontweight='bold', fontsize=10)
ax.set_xlabel('Number of Missing Values')
ax.set_title('Missing Values per Column', fontsize=14)
plt.tight_layout()
plt.show()

---

# 3. Target Variable Analysis

We examine the distribution of both target variables: **Dropout_Risk** and **Burnout_Level**.


## 3.1 Dropout Risk Distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
palette_dr = {'Yes': '#e74c3c', 'No': '#2ecc71'}
order_dr = ['Yes', 'No']
sns.countplot(x='Dropout_Risk', data=df, palette=palette_dr, order=order_dr, ax=axes[0], edgecolor='black')
axes[0].set_title('Dropout Risk Distribution', fontsize=14)
axes[0].set_xlabel('Dropout Risk')
axes[0].set_ylabel('Count')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}\n({p.get_height()/len(df)*100:.1f}%)',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontweight='bold', fontsize=11)

# Pie chart
dr_counts = df['Dropout_Risk'].value_counts().reindex(order_dr)
axes[1].pie(dr_counts, labels=order_dr, colors=[palette_dr[k] for k in order_dr],
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'},
            explode=[0.03]*len(order_dr), shadow=True)
axes[1].set_title('Dropout Risk Proportion', fontsize=14)

plt.tight_layout()
plt.show()

## 3.2 Burnout Level Distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

palette_bo = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}
order_bo = ['Low', 'Medium', 'High']

sns.countplot(x='Burnout_Level', data=df, palette=palette_bo, order=order_bo, ax=axes[0], edgecolor='black')
axes[0].set_title('Burnout Level Distribution', fontsize=14)
axes[0].set_xlabel('Burnout Level')
axes[0].set_ylabel('Count')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}\n({p.get_height()/len(df)*100:.1f}%)',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontweight='bold', fontsize=11)

bo_counts = df['Burnout_Level'].value_counts().reindex(order_bo)
axes[1].pie(bo_counts, labels=order_bo, colors=[palette_bo[k] for k in order_bo],
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'},
            explode=[0.03]*len(order_bo), shadow=True)
axes[1].set_title('Burnout Level Proportion', fontsize=14)

plt.tight_layout()
plt.show()

## 3.3 Cross-Tabulation: Dropout Risk × Burnout Level


In [ ]:
ct = pd.crosstab(df['Burnout_Level'], df['Dropout_Risk'], margins=True, margins_name='Total')
ct_pct = pd.crosstab(df['Burnout_Level'], df['Dropout_Risk'], normalize='all', margins=True, margins_name='Total') * 100

print("Counts:")
print(ct)
print("\nPercentages:")
print(ct_pct.round(1).astype(str) + '%')

In [ ]:
# Stacked bar: Burnout Level by Dropout Risk
ct_plot = pd.crosstab(df['Burnout_Level'], df['Dropout_Risk'], normalize='index') * 100
ct_plot = ct_plot.reindex(order_bo)

fig, ax = plt.subplots(figsize=(8, 5))
ct_plot.plot(kind='bar', stacked=True, ax=ax, color=[palette_dr['No'], palette_dr['Yes']], edgecolor='black')
ax.set_title('Dropout Risk Proportion by Burnout Level', fontsize=14)
ax.set_xlabel('Burnout Level')
ax.set_ylabel('Percentage (%)')
ax.legend(title='Dropout Risk', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---

# 4. Demographic EDA

Exploring age, gender, year of study, department, and residence type in relation to targets.


## 4.1 Age Distribution by Dropout Risk


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram / KDE
for risk, color in zip(['Yes', 'No'], ['#e74c3c', '#2ecc71']):
    subset = df[df['Dropout_Risk'] == risk]['Age'].dropna()
    axes[0].hist(subset, bins=15, alpha=0.6, label=f'Risk={risk}', color=color, edgecolor='white')
axes[0].set_title('Age Distribution by Dropout Risk', fontsize=14)
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Box plot
sns.boxplot(x='Dropout_Risk', y='Age', data=df, palette=palette_dr, order=order_dr, ax=axes[1])
axes[1].set_title('Age by Dropout Risk (Box Plot)', fontsize=14)

plt.tight_layout()
plt.show()

## 4.2 Gender Breakdown


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Overall gender
sns.countplot(x='Gender', data=df, palette='Set2', ax=axes[0], edgecolor='black', order=sorted(df['Gender'].unique()))
axes[0].set_title('Gender Distribution', fontsize=14)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}', (p.get_x()+p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontweight='bold')

# Gender x Dropout Risk
ct_gender_dr = pd.crosstab(df['Gender'], df['Dropout_Risk'], normalize='index') * 100
ct_gender_dr = ct_gender_dr.reindex(columns=['Yes', 'No'])
ct_gender_dr.plot(kind='bar', ax=axes[1], color=[palette_dr['Yes'], palette_dr['No']], edgecolor='black')
axes[1].set_title('Dropout Risk by Gender (%)', fontsize=14)
axes[1].set_ylabel('Percentage')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(title='Dropout Risk')

# Gender x Burnout Level
ct_gender_bo = pd.crosstab(df['Gender'], df['Burnout_Level'], normalize='index') * 100
ct_gender_bo = ct_gender_bo.reindex(columns=order_bo)
ct_gender_bo.plot(kind='bar', ax=axes[2], color=[palette_bo['Low'], palette_bo['Medium'], palette_bo['High']], edgecolor='black')
axes[2].set_title('Burnout Level by Gender (%)', fontsize=14)
axes[2].set_ylabel('Percentage')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)
axes[2].legend(title='Burnout Level')

plt.tight_layout()
plt.show()

## 4.3 Year of Study Distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall
year_order = sorted(df['Year_of_Study'].unique())
sns.countplot(x='Year_of_Study', data=df, palette='Set2', order=year_order, ax=axes[0], edgecolor='black')
axes[0].set_title('Year of Study Distribution', fontsize=14)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}', (p.get_x()+p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontweight='bold')

# By Dropout Risk
ct_year = pd.crosstab(df['Year_of_Study'], df['Dropout_Risk'], normalize='index') * 100
ct_year = ct_year.reindex(index=year_order, columns=['Yes', 'No'])
ct_year.plot(kind='bar', ax=axes[1], color=[palette_dr['Yes'], palette_dr['No']], edgecolor='black')
axes[1].set_title('Dropout Risk by Year of Study (%)', fontsize=14)
axes[1].set_ylabel('Percentage')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(title='Dropout Risk')

plt.tight_layout()
plt.show()

## 4.4 Department Distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

dept_order = df['Department'].value_counts().index.tolist()
sns.countplot(y='Department', data=df, palette='Set2', order=dept_order, ax=axes[0], edgecolor='black')
axes[0].set_title('Department Distribution', fontsize=14)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_width()):,}', (p.get_width(), p.get_y()+p.get_height()/2.),
                     ha='left', va='center', fontweight='bold')

ct_dept = pd.crosstab(df['Department'], df['Dropout_Risk'], normalize='index') * 100
ct_dept = ct_dept.reindex(index=dept_order, columns=['Yes', 'No'])
ct_dept.plot(kind='barh', stacked=False, ax=axes[1], color=[palette_dr['Yes'], palette_dr['No']], edgecolor='black')
axes[1].set_title('Dropout Risk by Department (%)', fontsize=14)
axes[1].set_xlabel('Percentage')
axes[1].legend(title='Dropout Risk')

plt.tight_layout()
plt.show()

## 4.5 Residence Type Distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

res_order = df['Residence_Type'].value_counts().index.tolist()
sns.countplot(x='Residence_Type', data=df, palette='Set2', order=res_order, ax=axes[0], edgecolor='black')
axes[0].set_title('Residence Type Distribution', fontsize=14)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}', (p.get_x()+p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontweight='bold')

ct_res = pd.crosstab(df['Residence_Type'], df['Dropout_Risk'], normalize='index') * 100
ct_res = ct_res.reindex(index=res_order, columns=['Yes', 'No'])
ct_res.plot(kind='bar', ax=axes[1], color=[palette_dr['Yes'], palette_dr['No']], edgecolor='black')
axes[1].set_title('Dropout Risk by Residence Type (%)', fontsize=14)
axes[1].set_ylabel('Percentage')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(title='Dropout Risk')

plt.tight_layout()
plt.show()

---

# 5. Academic Features

Analyzing attendance, study hours, GPA, and backlogs against both targets.


## 5.1 Attendance Percentage


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Violin plot by Dropout Risk
sns.violinplot(x='Dropout_Risk', y='Attendance_Percent', data=df, palette=palette_dr,
               order=order_dr, inner='quartile', ax=axes[0])
axes[0].set_title('Attendance % by Dropout Risk', fontsize=14)

# Box plot by Burnout Level
sns.boxplot(x='Burnout_Level', y='Attendance_Percent', data=df, palette=palette_bo,
            order=order_bo, ax=axes[1])
axes[1].set_title('Attendance % by Burnout Level', fontsize=14)

plt.tight_layout()
plt.show()

# Stats
print("\n📊 Attendance % by Dropout Risk:")
print(df.groupby('Dropout_Risk')['Attendance_Percent'].describe().round(2))
print("\n📊 Attendance % by Burnout Level:")
print(df.groupby('Burnout_Level')['Attendance_Percent'].describe().round(2))

## 5.2 Study Hours Per Day


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x='Dropout_Risk', y='Study_Hours_Per_Day', data=df, palette=palette_dr,
            order=order_dr, ax=axes[0])
axes[0].set_title('Study Hours/Day by Dropout Risk', fontsize=14)

sns.violinplot(x='Burnout_Level', y='Study_Hours_Per_Day', data=df, palette=palette_bo,
               order=order_bo, inner='quartile', ax=axes[1])
axes[1].set_title('Study Hours/Day by Burnout Level', fontsize=14)

plt.tight_layout()
plt.show()

print("\n📊 Study Hours by Dropout Risk:")
print(df.groupby('Dropout_Risk')['Study_Hours_Per_Day'].describe().round(2))
print("\n📊 Study Hours by Burnout Level:")
print(df.groupby('Burnout_Level')['Study_Hours_Per_Day'].describe().round(2))

## 5.3 Previous GPA


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.violinplot(x='Dropout_Risk', y='Previous_GPA', data=df, palette=palette_dr,
               order=order_dr, inner='quartile', ax=axes[0])
axes[0].set_title('Previous GPA by Dropout Risk', fontsize=14)

sns.boxplot(x='Burnout_Level', y='Previous_GPA', data=df, palette=palette_bo,
            order=order_bo, ax=axes[1])
axes[1].set_title('Previous GPA by Burnout Level', fontsize=14)

plt.tight_layout()
plt.show()

print("\n📊 GPA by Dropout Risk:")
print(df.groupby('Dropout_Risk')['Previous_GPA'].describe().round(2))
print("\n📊 GPA by Burnout Level:")
print(df.groupby('Burnout_Level')['Previous_GPA'].describe().round(2))

## 5.4 Backlogs


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x='Dropout_Risk', y='Backlogs', data=df, palette=palette_dr,
            order=order_dr, ax=axes[0])
axes[0].set_title('Backlogs by Dropout Risk', fontsize=14)

sns.violinplot(x='Burnout_Level', y='Backlogs', data=df, palette=palette_bo,
               order=order_bo, inner='quartile', ax=axes[1])
axes[1].set_title('Backlogs by Burnout Level', fontsize=14)

plt.tight_layout()
plt.show()

print("\n📊 Backlogs by Dropout Risk:")
print(df.groupby('Dropout_Risk')['Backlogs'].describe().round(2))
print("\n📊 Backlogs by Burnout Level:")
print(df.groupby('Burnout_Level')['Backlogs'].describe().round(2))

## 5.5 Academic Features Combined Overview


In [ ]:
academic_cols = ['Attendance_Percent', 'Study_Hours_Per_Day', 'Previous_GPA', 'Backlogs']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col in zip(axes.flatten(), academic_cols):
    sns.boxplot(x='Dropout_Risk', y=col, data=df, palette=palette_dr, order=order_dr, ax=ax)
    ax.set_title(f'{col} by Dropout Risk', fontsize=12)

plt.suptitle('Academic Features vs Dropout Risk', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

---

# 6. Lifestyle & Mental Health

Exploring sleep, screen time, exercise, stress, anxiety, motivation, and more.


## 6.1 Sleep Hours


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.violinplot(x='Dropout_Risk', y='Sleep_Hours', data=df, palette=palette_dr,
               order=order_dr, inner='quartile', ax=axes[0])
axes[0].set_title('Sleep Hours by Dropout Risk', fontsize=14)

sns.boxplot(x='Burnout_Level', y='Sleep_Hours', data=df, palette=palette_bo,
            order=order_bo, ax=axes[1])
axes[1].set_title('Sleep Hours by Burnout Level', fontsize=14)

plt.tight_layout()
plt.show()

## 6.2 Screen Time Hours


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x='Dropout_Risk', y='Screen_Time_Hours', data=df, palette=palette_dr,
            order=order_dr, ax=axes[0])
axes[0].set_title('Screen Time (hrs) by Dropout Risk', fontsize=14)

sns.violinplot(x='Burnout_Level', y='Screen_Time_Hours', data=df, palette=palette_bo,
               order=order_bo, inner='quartile', ax=axes[1])
axes[1].set_title('Screen Time (hrs) by Burnout Level', fontsize=14)

plt.tight_layout()
plt.show()

## 6.3 Exercise Frequency


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ct_exercise_dr = pd.crosstab(df['Exercise_Freq_Per_Week'], df['Dropout_Risk'], normalize='index') * 100
ct_exercise_dr = ct_exercise_dr.reindex(columns=['Yes', 'No'])
ct_exercise_dr.plot(kind='bar', ax=axes[0], color=[palette_dr['Yes'], palette_dr['No']], edgecolor='black')
axes[0].set_title('Dropout Risk by Exercise Frequency (%)', fontsize=14)
axes[0].set_ylabel('Percentage')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].legend(title='Dropout Risk')

ct_exercise_bo = pd.crosstab(df['Exercise_Freq_Per_Week'], df['Burnout_Level'], normalize='index') * 100
ct_exercise_bo = ct_exercise_bo.reindex(columns=order_bo)
ct_exercise_bo.plot(kind='bar', ax=axes[1], color=[palette_bo['Low'], palette_bo['Medium'], palette_bo['High']], edgecolor='black')
axes[1].set_title('Burnout Level by Exercise Frequency (%)', fontsize=14)
axes[1].set_ylabel('Percentage')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(title='Burnout Level')

plt.tight_layout()
plt.show()

## 6.4 Mental Health Features: Stress, Anxiety, Motivation, Peer Pressure


In [ ]:
mental_cols = ['Stress_Level', 'Anxiety_Score', 'Motivation_Score', 'Peer_Pressure_Score']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col in zip(axes.flatten(), mental_cols):
    sns.boxplot(x='Burnout_Level', y=col, data=df, palette=palette_bo, order=order_bo, ax=ax)
    ax.set_title(f'{col} by Burnout Level', fontsize=12)

plt.suptitle('Mental Health Features vs Burnout Level', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col in zip(axes.flatten(), mental_cols):
    sns.violinplot(x='Dropout_Risk', y=col, data=df, palette=palette_dr, order=order_dr, inner='quartile', ax=ax)
    ax.set_title(f'{col} by Dropout Risk', fontsize=12)

plt.suptitle('Mental Health Features vs Dropout Risk', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 6.5 Pairplot of Key Lifestyle & Mental Health Features


In [ ]:
lifestyle_features = ['Sleep_Hours', 'Screen_Time_Hours', 'Exercise_Freq_Per_Week',
                       'Stress_Level', 'Anxiety_Score', 'Motivation_Score']
pair_df = df[lifestyle_features + ['Burnout_Level']].dropna()

g = sns.pairplot(pair_df, hue='Burnout_Level', hue_order=order_bo,
                 palette=palette_bo, diag_kind='kde', plot_kws={'alpha': 0.5, 's': 20},
                 height=2.2)
g.fig.suptitle('Pairplot: Lifestyle & Mental Health by Burnout Level', y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

## 6.6 Correlation Heatmap for Lifestyle & Mental Health Features


In [ ]:
lifestyle_all = ['Sleep_Hours', 'Screen_Time_Hours', 'Exercise_Freq_Per_Week',
                  'Social_Activity_Score', 'Financial_Stress_Score', 'Family_Support_Score',
                  'Stress_Level', 'Anxiety_Score', 'Motivation_Score', 'Peer_Pressure_Score']

fig, ax = plt.subplots(figsize=(12, 9))
corr_sub = df[lifestyle_all].corr()
mask = np.triu(np.ones_like(corr_sub, dtype=bool))
sns.heatmap(corr_sub, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn_r',
            center=0, square=True, linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Heatmap: Lifestyle & Mental Health Features', fontsize=14)
plt.tight_layout()
plt.show()

---

# 7. Correlation Analysis

Full correlation analysis across all numeric features.


## 7.1 Encode Targets for Correlation


In [ ]:
# Create numeric encodings for correlation analysis
df_corr = df.copy()
df_corr['Dropout_Risk_Num'] = (df_corr['Dropout_Risk'] == 'Yes').astype(int)
df_corr['Burnout_Level_Num'] = df_corr['Burnout_Level'].map({'Low': 0, 'Medium': 1, 'High': 2})

print("Dropout_Risk_Num: Yes=1, No=0")
print("Burnout_Level_Num: Low=0, Medium=1, High=2")

## 7.2 Full Correlation Heatmap


In [ ]:
numeric_cols = df_corr.select_dtypes(include=[np.number]).columns.tolist()
# Remove Student_ID
if 'Student_ID' in numeric_cols:
    numeric_cols.remove('Student_ID')

fig, ax = plt.subplots(figsize=(18, 14))
full_corr = df_corr[numeric_cols].corr()
mask = np.triu(np.ones_like(full_corr, dtype=bool))
sns.heatmap(full_corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            annot_kws={'size': 8}, vmin=-1, vmax=1,
            cbar_kws={'label': 'Correlation Coefficient'})
ax.set_title('Full Correlation Heatmap of All Numeric Features', fontsize=16, pad=20)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

## 7.3 Correlation with Target Variables


In [ ]:
# Correlation of features with numeric targets
target_corr = df_corr[numeric_cols][['Dropout_Risk_Num', 'Burnout_Level_Num']].corr().drop(
    ['Dropout_Risk_Num', 'Burnout_Level_Num'], axis=0
)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Dropout Risk
sort_dr = target_corr['Dropout_Risk_Num'].sort_values(ascending=True)
colors_dr = ['#e74c3c' if v > 0 else '#2ecc71' for v in sort_dr.values]
axes[0].barh(sort_dr.index, sort_dr.values, color=colors_dr, edgecolor='white', height=0.6)
axes[0].set_title('Feature Correlation with Dropout Risk', fontsize=14)
axes[0].set_xlabel('Correlation Coefficient')
axes[0].axvline(x=0, color='black', linewidth=0.8)

# Burnout Level
sort_bo = target_corr['Burnout_Level_Num'].sort_values(ascending=True)
colors_bo = ['#e74c3c' if v > 0 else '#2ecc71' for v in sort_bo.values]
axes[1].barh(sort_bo.index, sort_bo.values, color=colors_bo, edgecolor='white', height=0.6)
axes[1].set_title('Feature Correlation with Burnout Level', fontsize=14)
axes[1].set_xlabel('Correlation Coefficient')
axes[1].axvline(x=0, color='black', linewidth=0.8)

plt.suptitle('Feature Importance by Correlation with Targets', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 7.4 Top Correlations (Absolute > 0.1 with Targets)


In [ ]:
# Print top features correlated with each target
for target_name, col in [('Dropout Risk', 'Dropout_Risk_Num'), ('Burnout Level', 'Burnout_Level_Num')]:
    print(f"\n🎯 Top features correlated with {target_name}:")
    corr_vals = target_corr[col].drop([col], errors='ignore').abs().sort_values(ascending=False)
    for feat, val in corr_vals.head(10).items():
        actual = target_corr.loc[feat, col]
        print(f"   {feat:30s}  r = {actual:+.3f}")

## 7.5 Interactive Correlation Plot (Plotly)


In [ ]:
# Interactive heatmap with Plotly
fig = px.imshow(
    full_corr.values,
    x=full_corr.columns,
    y=full_corr.index,
    color_continuous_scale='RdYlGn_r',
    zmin=-1, zmax=1,
    title='Interactive Correlation Heatmap',
    aspect='auto',
    text_auto='.2f'
)
fig.update_layout(
    width=900, height=800,
    font=dict(size=9),
    xaxis=dict(tickangle=45)
)
fig.show()

---

# 8. Key Insights

## Summary of Findings

### 📊 Data Overview
- The dataset contains **800 students** across **25 features** covering demographics, academics, lifestyle, and mental health.
- **612 missing values** are present across multiple columns, indicating the need for careful imputation before modeling.
- Dropout Risk is moderately imbalanced: **Yes (54.6%)** vs **No (45.4%)**.
- Burnout Level is skewed toward lower levels: **Low (40%)**, **Medium (35%)**, **High (25%)**.

### 🎓 Academic Features
- **Lower attendance, study hours, and GPA** are consistently associated with higher dropout risk and burnout.
- **More backlogs** correlate with both increased dropout risk and higher burnout levels.
- Students with high burnout tend to show lower GPAs and poorer attendance patterns.

### 🧠 Mental Health & Lifestyle
- **Higher stress and anxiety scores** show the strongest positive correlations with burnout and dropout risk.
- **Higher motivation scores** are protective — negatively correlated with both dropout risk and burnout.
- **Sleep deprivation and excessive screen time** are linked to higher burnout risk.
- **Regular exercise** appears to be a protective factor against both burnout and dropout.

### 👥 Demographics
- Dropout risk and burnout patterns vary by department, suggesting institutional factors play a role.
- Year of study shows interesting patterns — certain years may represent critical risk periods.
- Residence type may influence support systems and dropout risk.

### 🔗 Key Correlations
- **Stress Level** and **Anxiety Score** are among the strongest positive predictors for both targets.
- **Motivation Score** and **Family Support Score** are among the strongest negative (protective) predictors.
- **Financial Stress** shows a notable positive correlation with burnout and dropout risk.

### 💡 Recommendations for Modeling
- Focus on **stress, anxiety, motivation, financial stress, and family support** as key features.
- Consider **interaction effects** between academic performance and mental health scores.
- Address missing values carefully — strategies like MICE or KNN imputation may outperform simple mean/median.
- Use **class-weighted models** or **SMOTE** to handle the moderate class imbalance in Dropout_Risk.
